## Estado del contrato de variables

Este notebook se ha actualizado para el cubo final: usa VPD en lugar de días secos consecutivos y no depende de las variables redundantes retiradas. Debe ejecutarse de nuevo antes de interpretar resultados.

# Validación y exploración del datacubo histórico

Este notebook es de **solo lectura**: comprueba el datacubo EGIF + ERA5 + topografía + CORINE ya generado. No descarga datos ni ejecuta el pipeline.

**Contrato temporal:** desde 2019-01-01 hasta la última fecha registrada en el XML EGIF. La meteorología y sus acumulados corresponden al día T (sin desplazamiento temporal).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr

plt.style.use('default')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white'})
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
CUBE_PATH = ROOT / 'data/processed/datacube/galicia_1km.nc'
assert CUBE_PATH.exists(), f'No se encuentra el datacubo: {CUBE_PATH}'
print(f'Leyendo: {CUBE_PATH}')

In [ ]:
cube = xr.open_dataset(CUBE_PATH)
display(cube)
print('Atributos globales:')
for key, value in cube.attrs.items():
    print(f'  - {key}: {value}')

## 1. Contrato, dimensiones y esquema

In [ ]:
EXPECTED_START = pd.Timestamp('2019-01-01')
REQUIRED_VARIABLES = {
    'is_galicia', 'elevation_mean', 'slope_mean', 'mixed_forest',
    'temperature_max_12_18h', 'relative_humidity_min_12_18h',
    'wind_speed_max_12_18h', 'precipitation_sum_30d',
    'target_ignicion',
}
dates = pd.DatetimeIndex(cube.time.values)
assert dates.min() == EXPECTED_START
assert len(dates) == len(pd.date_range(dates.min(), dates.max()))
assert REQUIRED_VARIABLES.issubset(cube.data_vars), REQUIRED_VARIABLES - set(cube.data_vars)
assert cube['is_galicia'].dtype.kind in 'biu'

n_active = int(cube['is_galicia'].sum())
summary = pd.DataFrame({
    'medida': ['días', 'celdas eje y', 'celdas eje x', 'celdas Galicia', 'variables'],
    'valor': [len(dates), cube.sizes['y'], cube.sizes['x'], n_active, len(cube.data_vars)],
})
display(summary)
print('✓ Contrato espacial, temporal y variables mínimas correcto.')

In [ ]:
groups = {
    'topografía': ['elevation_mean', 'elevation_std', 'slope_mean', 'slope_std', 'elevation_mean', 'slope_std'],
    'orientación': [name for name in cube.data_vars if name.startswith('aspect_')],
    'CORINE': ['artificial', 'agriculture', 'broadleaf_forest', 'coniferous_forest', 'mixed_forest', 'scrub', 'open_spaces', 'wetlands', 'water', 'mixed_forest'],
    'calendario': ['year', 'month', 'iso_week', 'day_of_year', 'day_of_week', 'is_weekend', 'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos'],
    'meteorología': [name for name in cube.data_vars if name.startswith(('temperature_', 'relative_humidity_', 'wind_speed_', 'precipitation_', 'consecutive_'))],
    'resultado / auditoría': ['target_ignicion', 'burned_area_ha', 'large_fire_500ha'],
}
for group, names in groups.items():
    print(f'\n{group.upper()} ({len(names)}):')
    print(', '.join(names))

## 2. Validaciones de rango y completitud

In [ ]:
active = cube['is_galicia'] == 1
static_vars = groups['topografía'] + groups['orientación'] + groups['CORINE']
static_missing = {name: float((cube[name].isnull() & active).sum() / active.sum()) for name in static_vars}
display(pd.Series(static_missing, name='fracción_na').sort_values().to_frame())
assert max(static_missing.values()) < 0.01, 'Hay demasiados NaN en capas estáticas.'

meteo_checks = {
    'temperatura (°C)': ('temperature_min', -45, 55),
    'humedad relativa (%)': ('relative_humidity_min', 0, 100),
    'viento (m/s)': ('wind_speed_max', 0, 80),
    'precipitación (mm)': ('precipitation_sum', 0, 500),
    'días secos': ('vpd_mean', 0, 500),
}
range_rows = []
for label, (name, low, high) in meteo_checks.items():
    values = cube[name].where(active)
    observed_min, observed_max = float(values.min()), float(values.max())
    range_rows.append([label, name, observed_min, observed_max, low <= observed_min and observed_max <= high])
ranges = pd.DataFrame(range_rows, columns=['variable', 'nombre', 'mínimo', 'máximo', 'rango_plausible'])
display(ranges)
assert ranges['rango_plausible'].all(), 'Hay valores meteorológicos fuera de un rango físico plausible.'

for longer, shorter in [('precipitation_sum_3d', 'precipitation_sum'), ('precipitation_sum_7d', 'precipitation_sum_3d'), ('precipitation_sum_14d', 'precipitation_sum_7d'), ('precipitation_sum_30d', 'precipitation_sum_14d')]:
    invalid_fraction = float(((cube[longer] + 1e-6) < cube[shorter]).where(active).mean())
    print(f'{longer} ≥ {shorter}: fracción que incumple = {invalid_fraction:.6f}')
    assert invalid_fraction < 1e-6
print('✓ Capas estáticas completas y acumulados meteorológicos coherentes.')

## 3. Mapas de comprobación

In [ ]:
def plot_layer(data, title, cmap='viridis', vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(8, 8), facecolor='white')
    image = data.where(active).plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=True)
    ax.set_title(title)
    ax.set_xlabel('x (EPSG:3035)')
    ax.set_ylabel('y (EPSG:3035)')
    ax.set_aspect('equal')
    plt.show()

plot_layer(cube['elevation_mean'], 'Altitud media (m)', 'terrain')
plot_layer(cube['slope_mean'], 'Pendiente media (grados)', 'magma')
plot_layer(cube['mixed_forest'], 'Fracción de cobertura forestal', 'YlGn', 0, 1)

In [ ]:
MAP_DATE = pd.Timestamp('2023-08-20')
if MAP_DATE not in dates:
    MAP_DATE = dates[len(dates) // 2]
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True, facecolor='white')
layers = [
    ('temperature_max_12_18h', 'Temperatura máxima 12–18 h (°C)', 'RdYlBu_r'),
    ('relative_humidity_min_12_18h', 'Humedad relativa mínima 12–18 h (%)', 'YlGnBu'),
    ('precipitation_sum_30d', 'Precipitación acumulada 30 días (mm)', 'Blues'),
]
for ax, (name, label, cmap) in zip(axes, layers):
    cube[name].sel(time=MAP_DATE).where(active).plot(ax=ax, cmap=cmap, add_colorbar=True)
    ax.set_title(f'{label}\n{MAP_DATE.date()}')
    ax.set_aspect('equal')
plt.show()

## 4. Cobertura EGIF y distribución del target

In [ ]:
daily_ignitions = cube['target_ignicion'].where(active).sum(dim=('y', 'x'), skipna=True).to_series()
coverage = pd.DataFrame({'igniciones': daily_ignitions})
coverage['año'] = coverage.index.year
display(coverage.groupby('año')[['igniciones']].agg(['mean', 'sum']))

fig, ax = plt.subplots(figsize=(14, 4), facecolor='white')
coverage['igniciones'].plot(ax=ax, color='tab:red', linewidth=0.8)
ax.set(title='Celdas con ignición EGIF por día', ylabel='n.º de celdas', xlabel='fecha')
plt.show()

assert int((cube['target_ignicion'].isnull() & active).sum()) == 0
print('✓ Todo el intervalo del cubo está cubierto por EGIF; 0 significa ausencia de ignición.')

## Cierre

Si todas las celdas anteriores finalizan sin excepciones, el datacubo es consistente para la fase de modelado. Cerrar el dataset libera el archivo en Windows.

In [ ]:
cube.close()